

`pip install dotenv gradio openai`

In [1]:
# import the necessary libraries

import os
import glob
import tiktoken
import numpy as np
import plotly.graph_objects as go
import gradio as gr
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
from pathlib import Path
from openai import OpenAI

c:\Users\DELL\Documents\LLM Projects\q_transfer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\DELL\AppData\Local\Temp\ipykernel_18956\2272925242.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [2]:
# Constants

MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"

In [3]:
# Load the OpenAI API KEY from .env

load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print("OpenAI Key successfully loaded.")
else:
    print("OpenAI key not loaded.")


OpenAI Key successfully loaded.


In [4]:
# Create an instance of the OpenAI python client library

openai = OpenAI()

In [8]:
# Display the number of files and characters in the knowledge base

knowledge_base_path = "../q_transfer_knowledge_base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True) # recursive=True tells Python to search inside subfolders too, not just the main folder.
print(files)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, "r", encoding="utf-8") as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in the knowledge base: {len(entire_knowledge_base):,}")

['../q_transfer_knowledge_base\\README.md', '../q_transfer_knowledge_base\\company\\branches.md', '../q_transfer_knowledge_base\\company\\company-overview.md', '../q_transfer_knowledge_base\\company\\organization.md', '../q_transfer_knowledge_base\\compliance\\aml-cft.md', '../q_transfer_knowledge_base\\compliance\\regulatory-reporting.md', '../q_transfer_knowledge_base\\compliance\\sanctions-screening.md', '../q_transfer_knowledge_base\\compliance\\transaction-monitoring.md', '../q_transfer_knowledge_base\\customers\\complaints.md', '../q_transfer_knowledge_base\\customers\\customer-profiles.md', '../q_transfer_knowledge_base\\customers\\customer-support.md', '../q_transfer_knowledge_base\\customers\\kyc-and-verification.md', '../q_transfer_knowledge_base\\employees\\employee-records.md', '../q_transfer_knowledge_base\\employees\\onboarding-and-offboarding.md', '../q_transfer_knowledge_base\\employees\\roles-and-access.md', '../q_transfer_knowledge_base\\finance\\ledger-and-reconcilia

In [6]:
# print the entire knowledge base
print(entire_knowledge_base)

# Q-Transfer Synthetic Fintech Knowledge Base

This is a fictional knowledge base for building and testing a Retrieval-Augmented Generation (RAG) application for Q-Transfer, a fictional fintech company.

## Scope

The dataset covers:
- Company structure
- Branches
- Employees and access
- Customers and KYC
- Local and international transfers
- Fees and FX
- Transaction operations
- AML/CFT and sanctions
- Fraud and risk
- Investments
- Finance and reconciliation
- Technology and security
- Support
- Policies
- Third-party management

## Important

All people, customer records, transaction IDs, investment accounts, branches, products, targets, and company information in this dataset are synthetic.

## Suggested RAG Questions

### Direct retrieval
- What are the transaction states?
- What are the responsibilities of Treasury?
- What does KYC mean at Q-Transfer?
- What investment products does Q-Transfer offer?
- Which branches are in Nigeria?

### Multi-document questions
- What should h

In [24]:
# Tokenize the entire knowledge base and get the number of tokens

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
print(f"The number of {MODEL} converted to is: {len(tokens):,}")

The number of gpt-4.1-nano converted to is: 4,684


In [25]:
# display the tokens and their representations

for token in tokens:
    print(token, repr(encoding.decode([token])))

2 '#'
1486 ' Q'
12 '-'
19355 'Transfer'
119399 ' Synthetic'
7772 ' Fin'
16710 'tech'
42892 ' Knowledge'
8729 ' Base'
279 '\n\n'
2500 'This'
382 ' is'
261 ' a'
77507 ' fictional'
7124 ' knowledge'
3611 ' base'
395 ' for'
6282 ' building'
326 ' and'
11493 ' testing'
261 ' a'
156679 ' Retrieval'
118469 '-Aug'
34982 'mented'
32476 ' Generation'
350 ' ('
49 'R'
2971 'AG'
8 ')'
5200 ' application'
395 ' for'
1486 ' Q'
12 '-'
19355 'Transfer'
11 ','
261 ' a'
77507 ' fictional'
166727 ' fintech'
3175 ' company'
364 '.\n\n'
877 '##'
67040 ' Scope'
279 '\n\n'
976 'The'
20830 ' dataset'
17804 ' covers'
734 ':\n'
12 '-'
9709 ' Company'
8866 ' structure'
198 '\n'
12 '-'
42649 ' Branch'
268 'es'
198 '\n'
12 '-'
66628 ' Employees'
326 ' and'
3158 ' access'
198 '\n'
12 '-'
42953 ' Customers'
326 ' and'
658 ' K'
50069 'YC'
198 '\n'
12 '-'
11274 ' Local'
326 ' and'
7544 ' international'
45837 ' transfers'
198 '\n'
12 '-'
66163 ' Fees'
326 ' and'
46154 ' FX'
198 '\n'
12 '-'
29766 ' Transaction'
12084 ' o